# Final raster row-order and completion audit

This notebook is a read-only final verification for the existing Global dataset. It checks (1) MODIS `samples.parquet` physical row order against canonical candidates sorted exactly as the MODIS writer sorts `year_samples`, (2) Sentinel-1 row order and `row_index` against the writer's cluster/station/date sort, and (3) Zarr leading dimensions, representative reads, and per-row completion flags. It processes one year at a time to limit memory use.

It does not rewrite any data.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
import zarr
from IPython.display import display

ROOT = Path.home() / "Code/UG_SM/Data/Global"
YEARS = list(range(2015, 2027))
CANDIDATES = ROOT / "station_date_candidates.parquet"
STATION_CLUSTERS = ROOT / "sentinel1/stations_with_clusters.parquet"
MODIS_ROOT = ROOT / "modis/evi_lst_patch_dataset"
S1_ROOT = ROOT / "sentinel1/nearest_valid_overview_v1"
AUDIT_OUT = ROOT / "raster_row_order_audit.json"

required = [ROOT, CANDIDATES, STATION_CLUSTERS, MODIS_ROOT, S1_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Required paths missing:\n" + "\n".join(missing))

con = duckdb.connect(":memory:")
con.execute("SET threads=2")
con.execute("SET memory_limit='512MB'")
# Sorts happen one year at a time. DuckDB spills to this local temp directory if needed.
TMP = ROOT / "_raster_audit_tmp"
TMP.mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = '" + str(TMP).replace("'", "''") + "'")


def sqlq(path):
    return "'" + Path(path).as_posix().replace("'", "''") + "'"


def normalize_identity(df, include_cluster=False, include_row_index=False):
    df = df.copy().reset_index(drop=True)
    df["sample_id"] = pd.to_numeric(df["sample_id"], errors="raise").astype("int64")
    df["station_id"] = df["station_id"].astype("string").str.strip()
    df["sample_date"] = pd.to_datetime(df["sample_date"], errors="raise").dt.strftime("%Y-%m-%d")
    if include_cluster:
        df["cluster_id"] = pd.to_numeric(df["cluster_id"], errors="raise").astype("int64")
    if include_row_index:
        df["row_index"] = pd.to_numeric(df["row_index"], errors="raise").astype("int64")
    cols = ["sample_id", "station_id", "sample_date"]
    if include_cluster:
        cols.append("cluster_id")
    if include_row_index:
        cols.append("row_index")
    return df[cols]


def compare_in_order(expected, actual, columns):
    expected = expected[columns].reset_index(drop=True)
    actual = actual[columns].reset_index(drop=True)
    if len(expected) != len(actual):
        common = min(len(expected), len(actual))
        limit_note = f"length mismatch expected={len(expected):,}, actual={len(actual):,}"
    else:
        common = len(expected)
        limit_note = None
    mismatch_positions = np.array([], dtype=np.int64)
    if common:
        diff = np.zeros(common, dtype=bool)
        for col in columns:
            a = expected[col].iloc[:common].to_numpy()
            b = actual[col].iloc[:common].to_numpy()
            diff |= (a != b)
        mismatch_positions = np.flatnonzero(diff)
    passed = len(expected) == len(actual) and len(mismatch_positions) == 0
    examples = []
    for ix in mismatch_positions[:5].tolist():
        examples.append({
            "position": int(ix),
            "expected": {c: str(expected.at[ix, c]) for c in columns},
            "actual": {c: str(actual.at[ix, c]) for c in columns},
        })
    return passed, int(len(mismatch_positions)), limit_note, examples

print("Audit root:", ROOT)
print("Canonical rows:", con.execute(f"SELECT COUNT(*) FROM read_parquet({sqlq(CANDIDATES)})").fetchone()[0])

Audit root: /home/mayur/Code/UG_SM/Data/Global
Canonical rows: 3514643


In [2]:
results = []

for year in YEARS:
    print(f"\n===== {year} =====")
    # ------------------------------------------------------------
    # MODIS: exact expected physical order = sample_date, sample_id
    # as used in MODIS_GLOBAL_EVI.ipynb before writing both arrays and metadata.
    # ------------------------------------------------------------
    modis_sidecar = MODIS_ROOT / str(year) / "samples.parquet"
    modis_store = MODIS_ROOT / str(year) / "patches.zarr"
    m = {"year": year, "modis_order_match": False, "modis_shapes_match": False,
         "modis_all_done": False, "modis_reads_ok": False, "modis_error": None}
    try:
        expected_m = con.execute(f'''
            SELECT CAST(sample_id AS BIGINT) AS sample_id,
                   TRIM(CAST(station_id AS VARCHAR)) AS station_id,
                   CAST(sample_date AS DATE) AS sample_date
            FROM read_parquet({sqlq(CANDIDATES)})
            WHERE YEAR(CAST(sample_date AS DATE)) = {year}
            ORDER BY CAST(sample_date AS DATE), CAST(sample_id AS BIGINT)
        ''').fetchdf()
        actual_m = pd.read_parquet(modis_sidecar, columns=["sample_id", "station_id", "sample_date"])
        expected_m = normalize_identity(expected_m)
        actual_m = normalize_identity(actual_m)
        mpass, mcount, mlen, mexamples = compare_in_order(
            expected_m, actual_m, ["sample_id", "station_id", "sample_date"]
        )
        m["modis_order_match"] = bool(mpass)
        m["modis_rows_expected"] = int(len(expected_m))
        m["modis_rows_sidecar"] = int(len(actual_m))
        m["modis_mismatch_rows_first_common"] = int(mcount)
        m["modis_length_note"] = mlen
        m["modis_mismatch_examples"] = mexamples

        mg = zarr.open_group(str(modis_store), mode="r")
        m_arrays = {name: arr for name, arr in mg.arrays()}
        m_n = len(expected_m)
        m["modis_array_shapes"] = {name: list(arr.shape) for name, arr in m_arrays.items()}
        m["modis_shapes_match"] = bool(
            all(len(arr.shape) > 0 and int(arr.shape[0]) == m_n for arr in m_arrays.values())
            and all(k in m_arrays for k in ["evi", "lst", "done"])
        )
        done = np.asarray(m_arrays["done"][:], dtype=bool)
        m["modis_done_true"] = int(done.sum())
        m["modis_done_total"] = int(done.size)
        m["modis_all_done"] = bool(done.all())
        indices = sorted(set([0, m_n // 2, m_n - 1])) if m_n else []
        for name, arr in m_arrays.items():
            for ix in indices:
                v = np.asarray(arr[ix])
                if v.size == 0:
                    raise ValueError(f"Empty read from {name}[{ix}]")
        m["modis_reads_ok"] = True
    except Exception as exc:
        m["modis_error"] = f"{type(exc).__name__}: {exc}"
    results.append(m)
    print("MODIS:", {k: m.get(k) for k in ["modis_order_match", "modis_shapes_match", "modis_all_done", "modis_reads_ok", "modis_error"]})

    # ------------------------------------------------------------
    # Sentinel-1: writer sort = cluster_id, station_id, sample_date;
    # row_index is assigned as physical row positions 0..N-1.
    # ------------------------------------------------------------
    s1_sidecar = S1_ROOT / f"samples_nearest_valid_{year}.parquet"
    s1_store = S1_ROOT / f"s1_nearest_valid_{year}.zarr"
    s = {"year": year, "s1_order_match": False, "s1_row_index_match": False,
         "s1_shapes_match": False, "s1_all_done": False, "s1_reads_ok": False,
         "s1_complete_attr": False, "s1_error": None}
    try:
        expected_s = con.execute(f'''
            SELECT CAST(c.sample_id AS BIGINT) AS sample_id,
                   TRIM(CAST(c.station_id AS VARCHAR)) AS station_id,
                   CAST(c.sample_date AS DATE) AS sample_date,
                   CAST(sc.cluster_id AS BIGINT) AS cluster_id
            FROM read_parquet({sqlq(CANDIDATES)}) AS c
            JOIN read_parquet({sqlq(STATION_CLUSTERS)}) AS sc USING (station_id)
            WHERE YEAR(CAST(c.sample_date AS DATE)) = {year}
            ORDER BY CAST(sc.cluster_id AS BIGINT),
                     TRIM(CAST(c.station_id AS VARCHAR)),
                     CAST(c.sample_date AS DATE)
        ''').fetchdf()
        expected_s = normalize_identity(expected_s, include_cluster=True)
        expected_s["row_index"] = np.arange(len(expected_s), dtype=np.int64)
        actual_s_raw = pd.read_parquet(
            s1_sidecar,
            columns=["sample_id", "station_id", "sample_date", "cluster_id", "row_index"],
        )
        actual_s = normalize_identity(actual_s_raw, include_cluster=True, include_row_index=True)
        s_cols = ["sample_id", "station_id", "sample_date", "cluster_id", "row_index"]
        spass, scount, slen, sexamples = compare_in_order(expected_s, actual_s, s_cols)
        s["s1_order_match"] = bool(spass)
        s["s1_rows_expected"] = int(len(expected_s))
        s["s1_rows_sidecar"] = int(len(actual_s))
        s["s1_mismatch_rows_first_common"] = int(scount)
        s["s1_length_note"] = slen
        s["s1_mismatch_examples"] = sexamples
        row_index = actual_s["row_index"].to_numpy(dtype=np.int64)
        s["s1_row_index_match"] = bool(
            len(row_index) == len(expected_s)
            and np.array_equal(row_index, np.arange(len(row_index), dtype=np.int64))
        )

        sg = zarr.open_group(str(s1_store), mode="r")
        s_arrays = {name: arr for name, arr in sg.arrays()}
        s_n = len(expected_s)
        s["s1_array_shapes"] = {name: list(arr.shape) for name, arr in s_arrays.items()}
        s["s1_shapes_match"] = bool(
            all(len(arr.shape) > 0 and int(arr.shape[0]) == s_n for arr in s_arrays.values())
            and "sample_done" in s_arrays
        )
        done = np.asarray(s_arrays["sample_done"][:], dtype=bool)
        s["s1_done_true"] = int(done.sum())
        s["s1_done_total"] = int(done.size)
        s["s1_all_done"] = bool(done.all())
        s["s1_complete_attr"] = bool(sg.attrs.get("complete", False))
        indices = sorted(set([0, s_n // 2, s_n - 1])) if s_n else []
        for name, arr in s_arrays.items():
            for ix in indices:
                v = np.asarray(arr[ix])
                if v.size == 0:
                    raise ValueError(f"Empty read from {name}[{ix}]")
        s["s1_reads_ok"] = True
    except Exception as exc:
        s["s1_error"] = f"{type(exc).__name__}: {exc}"
    results.append(s)
    print("Sentinel-1:", {k: s.get(k) for k in ["s1_order_match", "s1_row_index_match", "s1_shapes_match", "s1_all_done", "s1_complete_attr", "s1_reads_ok", "s1_error"]})

con.close()
summary = pd.DataFrame(results)
display(summary)
summary.to_json(AUDIT_OUT, orient="records", indent=2)
print("\nWrote audit report:", AUDIT_OUT)

# Final, explicit verdicts. A failing condition remains visible in `summary` above.
modis_pass = all(
    bool(r.get("modis_order_match")) and bool(r.get("modis_shapes_match"))
    and bool(r.get("modis_all_done")) and bool(r.get("modis_reads_ok"))
    for r in results if "modis_order_match" in r
)
s1_rows = [r for r in results if "s1_order_match" in r]
s1_pass = all(
    bool(r.get("s1_order_match")) and bool(r.get("s1_row_index_match"))
    and bool(r.get("s1_shapes_match")) and bool(r.get("s1_all_done"))
    and bool(r.get("s1_complete_attr")) and bool(r.get("s1_reads_ok"))
    for r in s1_rows
)
print("\nFINAL MODIS ROW-ORDER / COMPLETENESS:", "PASS" if modis_pass else "FAIL")
print("FINAL SENTINEL-1 ROW-ORDER / COMPLETENESS:", "PASS" if s1_pass else "FAIL")
if not modis_pass or not s1_pass:
    print("Review the corresponding error, mismatch examples, or done counts in `summary` before using raster inputs.")


===== 2015 =====
MODIS: {'modis_order_match': True, 'modis_shapes_match': True, 'modis_all_done': True, 'modis_reads_ok': True, 'modis_error': None}


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


Sentinel-1: {'s1_order_match': True, 's1_row_index_match': True, 's1_shapes_match': True, 's1_all_done': True, 's1_complete_attr': True, 's1_reads_ok': True, 's1_error': None}

===== 2016 =====
MODIS: {'modis_order_match': True, 'modis_shapes_match': True, 'modis_all_done': True, 'modis_reads_ok': True, 'modis_error': None}


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


Sentinel-1: {'s1_order_match': True, 's1_row_index_match': True, 's1_shapes_match': True, 's1_all_done': True, 's1_complete_attr': True, 's1_reads_ok': True, 's1_error': None}

===== 2017 =====
MODIS: {'modis_order_match': True, 'modis_shapes_match': True, 'modis_all_done': True, 'modis_reads_ok': True, 'modis_error': None}


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


... (16 outputs left out)

Sentinel-1: {'s1_order_match': True, 's1_row_index_match': True, 's1_shapes_match': True, 's1_all_done': True, 's1_complete_attr': True, 's1_reads_ok': True, 's1_error': None}

===== 2026 =====
MODIS: {'modis_order_match': True, 'modis_shapes_match': True, 'modis_all_done': True, 'modis_reads_ok': True, 'modis_error': None}
Sentinel-1: {'s1_order_match': True, 's1_row_index_match': True, 's1_shapes_match': True, 's1_all_done': True, 's1_complete_attr': True, 's1_reads_ok': True, 's1_error': None}


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


,year,modis_order_match,modis_shapes_match,modis_all_done,modis_reads_ok,modis_error,modis_rows_expected,modis_rows_sidecar,modis_mismatch_rows_first_common,modis_length_note,...,s1_complete_attr,s1_error,s1_rows_expected,s1_rows_sidecar,s1_mismatch_rows_first_common,s1_length_note,s1_mismatch_examples,s1_array_shapes,s1_done_true,s1_done_total
0,2015,True,True,True,True,NaN,329903.0,329903.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,NaN,329903.0,329903.0,0.0,NaN,[],"{'hh': [329903, 13, 13], 'vh': [329903, 13, 13...",329903.0,329903.0
2,2016,True,True,True,True,NaN,394790.0,394790.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,NaN,394790.0,394790.0,0.0,NaN,[],"{'hv': [394790, 13, 13], 'vv': [394790, 13, 13...",394790.0,394790.0
4,2017,True,True,True,True,NaN,369953.0,369953.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,NaN,369953.0,369953.0,0.0,NaN,[],"{'vv': [369953, 13, 13], 'obs_count': [369953,...",369953.0,369953.0
6,2018,True,True,True,True,NaN,344806.0,344806.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,NaN,344806.0,344806.0,0.0,NaN,[],"{'source_day_offset': [344806, 2], 'sample_don...",344806.0,344806.0
8,2019,True,True,True,True,NaN,308679.0,308679.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,NaN,308679.0,308679.0,0.0,NaN,[],"{'source_day_offset': [308679, 2], 'obs_count'...",308679.0,308679.0



Wrote audit report: /home/mayur/Code/UG_SM/Data/Global/raster_row_order_audit.json

FINAL MODIS ROW-ORDER / COMPLETENESS: PASS
FINAL SENTINEL-1 ROW-ORDER / COMPLETENESS: PASS


## Interpreting the result

- A `*_order_match = True` result means the physical sidecar row sequence matches the deterministic order used by the inspected producer notebook.
- `*_all_done = True` means the per-sample completion flag is true for every row; this is stronger than a row-count check.
- `*_shapes_match = True` and `*_reads_ok = True` confirm the expected row dimension and successful sample reads.
- The audit is read-only except for `raster_row_order_audit.json` and its temporary DuckDB directory. It does not modify the source Parquet or Zarr stores.
- This sequence/completion audit, together with the original writer logic, checks the sample-to-row relationship. It cannot independently reconstruct pixel provenance if a Zarr store was manually reordered or overwritten outside the inspected pipeline.